# 02 · KVPI vs DEM – Validación Topográfica
### KiwiTrackingIA · Lote "El Abrojito" · 2019–2025

**Descripción:**  
Validación topográfica del KVPI mediante cruce con DEM Copernicus y raster de pendiente.  
Evalúa si la variabilidad espacial del potencial productivo está estructurada por la topografía del lote.

**Inputs requeridos:**
- `datos/KVPI_serie_2019_2025.csv` — generado por `01_KVPI_Calculo.ipynb`
- `datos/DEM_Miramar_UTM.tif` — DEM Copernicus reproyectado a UTM
- `datos/Slope_Recalc_UTM.tif` — raster de pendiente derivado del DEM

**Outputs generados:**
- Gráficos de dispersión KVPI vs elevación / pendiente
- Boxplots por clase de KVPI
- Tabla cruzada KVPI × elevación
- Mapa Folium de ambientes productivos estructurales


## 1 · Librerías y configuración

In [ ]:
# ============================================================
# KiwiTrackingIA – 02_KVPI_vs_DEM.ipynb
# Autor: Darío Nicolás Sánchez Leguizamón
# Fecha: 2026-03-30
# Licencia: MIT
# Repositorio: https://github.com/sdarionicolas-boop/KiwiTrackingIA
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
import rasterio
import json
from shapely.geometry import Point
import folium

plt.style.use("default")
print("✅ Librerías cargadas correctamente")

## 2 · Carga de datos

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_PATH = "/content/drive/MyDrive/KiwiTrackingIA"

# Serie KVPI (output del notebook 01)
csv_kvpi = f"{DRIVE_PATH}/KVPI_serie_2019_2025.csv"
df_kvpi  = pd.read_csv(csv_kvpi)

print("📊 Dimensión KVPI:", df_kvpi.shape)
print("📋 Columnas:", list(df_kvpi.columns))
display(df_kvpi.head())

## 3 · Construcción del GeoDataFrame

In [ ]:
def geojson_to_point(geojson_str):
    """Extrae geometría Point desde la columna .geo exportada por GEE."""
    geo = json.loads(geojson_str)
    return Point(geo["coordinates"])

gdf_kvpi = gpd.GeoDataFrame(
    df_kvpi,
    geometry=df_kvpi[".geo"].apply(geojson_to_point),
    crs="EPSG:4326"
)

print("🌍 GeoDataFrame creado:", len(gdf_kvpi), "registros")
gdf_kvpi.head()

## 4 · Carga de rasters y extracción de valores topográficos

In [ ]:
dem_path   = f"{DRIVE_PATH}/DEM_Miramar_UTM.tif"
slope_path = f"{DRIVE_PATH}/Slope_Recalc_UTM.tif"

dem   = rasterio.open(dem_path)
slope = rasterio.open(slope_path)

print("📐 CRS del DEM:", dem.crs)

# Reproyectar puntos al CRS del DEM (UTM)
gdf_kvpi_utm = gdf_kvpi.to_crs(dem.crs)
print("📐 CRS puntos reproyectados:", gdf_kvpi_utm.crs)

In [ ]:
# Extraer elevación y pendiente por punto
coords       = [(geom.x, geom.y) for geom in gdf_kvpi_utm.geometry]
elev_values  = [val[0] for val in dem.sample(coords)]
slope_values = [val[0] for val in slope.sample(coords)]

gdf_kvpi_utm["elev_m"] = elev_values
gdf_kvpi_utm["slope"]  = slope_values

print("✅ Elevación y pendiente extraídas")
gdf_kvpi_utm[["KVPI", "elev_m", "slope"]].describe()

## 5 · Análisis exploratorio: KVPI vs elevación y pendiente

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(gdf_kvpi_utm["elev_m"], gdf_kvpi_utm["KVPI"], alpha=0.6, s=25)
axes[0].set_xlabel("Elevación (m)"); axes[0].set_ylabel("KVPI")
axes[0].set_title("KVPI vs Elevación"); axes[0].grid(True)

axes[1].scatter(gdf_kvpi_utm["slope"], gdf_kvpi_utm["KVPI"], alpha=0.6, s=25, color='darkorange')
axes[1].set_xlabel("Pendiente (%)"); axes[1].set_ylabel("KVPI")
axes[1].set_title("KVPI vs Pendiente"); axes[1].grid(True)

plt.tight_layout()
plt.show()

## 6 · Clasificación KVPI en terciles y estadísticos por clase

In [ ]:
gdf_kvpi_utm["clase_kvpi"] = pd.qcut(
    gdf_kvpi_utm["KVPI"], q=3, labels=["Bajo", "Medio", "Alto"]
)

print("📊 Conteo por clase:")
print(gdf_kvpi_utm["clase_kvpi"].value_counts())

print("\n📊 Estadísticos de elevación por clase KVPI:")
display(gdf_kvpi_utm.groupby("clase_kvpi")["elev_m"].describe())

print("\n📊 Estadísticos de pendiente por clase KVPI:")
display(gdf_kvpi_utm.groupby("clase_kvpi")["slope"].describe())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

gdf_kvpi_utm.boxplot(column="elev_m", by="clase_kvpi", ax=axes[0], grid=False)
axes[0].set_title("Elevación por clase de KVPI")
axes[0].set_xlabel("Clase KVPI"); axes[0].set_ylabel("Elevación (m)")

gdf_kvpi_utm.boxplot(column="slope", by="clase_kvpi", ax=axes[1], grid=False)
axes[1].set_title("Pendiente por clase de KVPI")
axes[1].set_xlabel("Clase KVPI"); axes[1].set_ylabel("Pendiente (%)")

for ax in axes:
    ax.title.set_size(12)

plt.suptitle("")
plt.tight_layout()
plt.show()

## 7 · Tabla cruzada KVPI × Elevación

In [ ]:
gdf_kvpi_utm["clase_elev"] = pd.qcut(
    gdf_kvpi_utm["elev_m"], q=3, labels=["Baja", "Media", "Alta"]
)

tabla_cruce = pd.crosstab(
    gdf_kvpi_utm["clase_kvpi"],
    gdf_kvpi_utm["clase_elev"],
    normalize="index"
) * 100

print("📊 Tabla cruzada KVPI × Elevación (% por fila):")
display(tabla_cruce.round(1))

In [ ]:
# KVPI medio por ambiente
ambientes = (
    gdf_kvpi_utm
    .groupby(["clase_elev", "clase_kvpi"])
    .agg(KVPI_mean=("KVPI", "mean"), n=("KVPI", "count"))
    .reset_index()
)
display(ambientes)

## 8 · Clasificación de ambientes productivos estructurales

In [ ]:
def definir_ambiente(row):
    if row["clase_kvpi"] == "Bajo":
        return "Ambiente Limitante"
    elif row["clase_kvpi"] == "Medio":
        return "Ambiente Transicional"
    else:
        return "Ambiente Alto Potencial"

gdf_kvpi_utm["ambiente"] = gdf_kvpi_utm.apply(definir_ambiente, axis=1)

print("🏞️  Distribución de ambientes:")
print(gdf_kvpi_utm["ambiente"].value_counts())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
gdf_kvpi_utm.plot(
    column="ambiente", categorical=True, legend=True,
    cmap="RdYlGn", markersize=25, alpha=0.85, ax=ax
)
ax.set_title("Ambientes productivos estructurales – El Abrojito", fontsize=13)
ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Scatter coloreado por ambiente
color_amb = {"Ambiente Limitante": "red",
             "Ambiente Transicional": "orange",
             "Ambiente Alto Potencial": "green"}

plt.figure(figsize=(7, 6))
for amb, grp in gdf_kvpi_utm.groupby("ambiente"):
    plt.scatter(grp["elev_m"], grp["KVPI"], label=amb,
                color=color_amb[amb], alpha=0.65, s=30)
plt.xlabel("Elevación (m)"); plt.ylabel("KVPI")
plt.title("Relación KVPI – Elevación (por ambiente)")
plt.legend(); plt.grid(True)
plt.tight_layout(); plt.show()

## 9 · Mapa interactivo de ambientes

In [ ]:
gdf_web = gdf_kvpi_utm.to_crs(epsg=4326)
centro  = [gdf_web.geometry.y.mean(), gdf_web.geometry.x.mean()]

m = folium.Map(location=centro, zoom_start=16, tiles="OpenStreetMap")

color_map = {
    "Ambiente Alto Potencial": "green",
    "Ambiente Transicional":   "orange",
    "Ambiente Limitante":      "red"
}

for _, row in gdf_web.iterrows():
    tooltip = f"""
    <b>Planta ID:</b> {row['id']}<br>
    <b>Ambiente:</b> {row['ambiente']}<br>
    <b>KVPI:</b> {row['KVPI']:.3f}<br>
    <b>Elevación:</b> {row['elev_m']:.1f} m<br>
    <b>Pendiente:</b> {row['slope']:.1f} %
    """
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=5,
        color=color_map[row["ambiente"]],
        fill=True, fill_opacity=0.85,
        tooltip=tooltip
    ).add_to(m)

legend_html = """
<div style="position:fixed;bottom:40px;left:40px;width:310px;
            background:white;border:2px solid grey;z-index:9999;
            font-size:13px;padding:10px;border-radius:6px;">
<b>Ambientes productivos – Kiwi</b><br><br>
<span style="color:green;">●</span> <b>Ambiente Alto Potencial</b><br>
Mayor estabilidad productiva relativa.<br><br>
<span style="color:orange;">●</span> <b>Ambiente Transicional</b><br>
Zonas intermedias, sensibles al manejo.<br><br>
<span style="color:red;">●</span> <b>Ambiente Limitante</b><br>
Menor potencial relativo; posibles limitantes de suelo o drenaje.<br><br>
<i>Generado a partir de índices satelitales multianuales y topografía.</i>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

out_html = f"{DRIVE_PATH}/Mapa_Ambientes_ElAbrojito.html"
m.save(out_html)
print("✅ Mapa guardado en:", out_html)
m